# 如何使用 Claude 3 和 MongoDB 构建 RAG 系统


本教程实现了一个被提示扮演风险投资科技分析师角色的聊天机器人。该聊天机器人是一个朴素的 RAG 系统，以一批科技新闻文章作为其知识来源。本笔记本涵盖以下内容：

1. 按照完整教程搭建开发环境，从安装必要的库到配置 MongoDB 数据库。
2. 学习高效的数据处理方法，包括创建向量搜索索引，以及为数据摄取和查询处理准备数据。
3. 了解如何在 RAG 系统中使用 Claude 3 模型，基于从数据库检索到的上下文信息生成精准响应。


你需要准备以下内容：
- Claude API 密钥
- VoyageAI API 密钥
- Hugging Face 访问密钥

## 第 1 步：安装库、加载并准备数据


下面简要说明实现代码中使用的工具和库：
- anthropic：Anthropic 官方 Python 库，可访问最先进的语言模型。该库提供对 Claude 3 系列模型的访问，这些模型能够理解文本和图像。
- datasets：该库是 Hugging Face 生态系统的一部分。安装 'datasets' 后，我们可以使用多个已预处理、开箱即用的数据集，这些数据集对于训练和微调机器学习模型或对其性能做基准测试至关重要。
- pandas：这个数据科学库提供了用于数据操作、处理和分析的强大数据结构与方法。
- voyageai：这是访问 VoyageAI 系列嵌入模型的官方 Python 客户端库。
- pymongo：PyMongo 是 MongoDB 的 Python 工具包，用于与 MongoDB 数据库交互。



In [ ]:
%pip install pymongo datasets pandas anthropic voyageai

下面的代码片段执行以下步骤：
1. 导入必要的库：
- `os` 用于与操作系统交互，
- `requests` 用于发起 HTTP 请求，
- io 模块中的 `BytesIO` 用于在内存中把字节对象当作文件处理，
- `pandas`（以 pd 别名导入）用于数据操作和分析，以及
- google.colab 中的 `userdata` 用于访问存储在 Google Colab 密钥中的环境变量。
2. 函数定义：定义了带有两个参数的 `download_and_combine_parquet_files` 函数：
- `parquet_file_urls`：由 URL 字符串组成的列表，每个 URL 指向一个 Parquet 文件，其中包含 tech-news-embedding 数据集的一个子集合。
- `hf_token`：表示 Hugging Face 授权密钥的字符串。访问密钥可以在 [Hugging Face 平台](https://huggingface.co/docs/hub/en/security-tokens#:~:text=To%20create%20an%20access%20token,you%27re%20ready%20to%20go!) 上创建或复制。
3. 下载并读取 Parquet 文件：该函数遍历 parquet_file_urls 中的每个 URL。对每个 URL，它会：
- 使用 requests.get 方法发起 GET 请求，传入 URL 和用于授权的请求头。
- 检查响应状态码是否为 200（OK），以确认请求成功。
- 如果成功，则把响应内容读入 BytesIO 对象（以便在内存中当作文件处理），再用 pandas.read_parquet 把该对象中的 Parquet 文件读入 Pandas DataFrame。
- 把该 DataFrame 追加到 all_dataframes 列表中。
4. 合并 DataFrame：在所有 Parquet 文件下载并读入 DataFrame 之后，代码会检查 `all_dataframes` 是否为空。如果存在可处理的 DataFrame，就用 pd.concat 把所有 DataFrame 拼接为一个 DataFrame，并使用 ignore_index=True 为合并后的新 DataFrame 重建索引。这个合并后的 DataFrame 就是 `download_and_combine_parquet_files` 函数的整体输出。

In [ ]:
from io import BytesIO

import pandas as pd
import requests
from google.colab import userdata


def download_and_combine_parquet_files(parquet_file_urls, hf_token):
    """
    Downloads Parquet files from the provided URLs using the given Hugging Face token,
    and returns a combined DataFrame.

    Parameters:
    - parquet_file_urls: List of strings, URLs to the Parquet files.
    - hf_token: String, Hugging Face authorization token.

    Returns:
    - combined_df: A pandas DataFrame containing the combined data from all Parquet files.
    """
    headers = {"Authorization": f"Bearer {hf_token}"}
    all_dataframes = []

    for parquet_file_url in parquet_file_urls:
        response = requests.get(parquet_file_url, headers=headers, timeout=60)
        if response.status_code == 200:
            parquet_bytes = BytesIO(response.content)
            df = pd.read_parquet(parquet_bytes)
            all_dataframes.append(df)
        else:
            print(
                f"Failed to download Parquet file from {parquet_file_url}: {response.status_code}"
            )

    if all_dataframes:
        combined_df = pd.concat(all_dataframes, ignore_index=True)
        return combined_df
    else:
        print("No dataframes to concatenate.")
        return None

下面是本教程所需的 Parquet 文件列表。全部文件的完整列表见[这里](https://huggingface.co/datasets/MongoDB/tech-news-embeddings/tree/refs%2Fconvert%2Fparquet/default/train)。每个 Parquet 文件约包含 45,000 个数据点。

在下面的代码片段中，tech-news-embeddings 数据集的一个子集被归集为一个 DataFrame，并赋值给变量 `combined_df`。


In [3]:
# Uncomment the links below to load more data
# For the full list of data visit: https://huggingface.co/datasets/MongoDB/tech-news-embeddings/tree/refs%2Fconvert%2Fparquet/default/train
parquet_files = [
    "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0000.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0001.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0002.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0003.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0004.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0005.parquet",
]

hf_token = userdata.get("HF_TOKEN")
combined_df = download_and_combine_parquet_files(parquet_files, hf_token)

作为数据准备的最后阶段，下面的代码片段展示了从归集后的数据集中移除 `_id` 列的步骤，因为本教程的后续步骤不需要该列。此外，每个数据点 embedding 列中的数据会从 numpy 数组转换为 Python 列表，以避免数据摄取过程中出现数据类型不兼容的错误。


In [4]:
# Remove the _id coloum from the intital dataset
combined_df = combined_df.drop(columns=["_id"])

# Remove the initial embedding coloumn as we are going to create new embeddings with VoyageAI embedding model
combined_df = combined_df.drop(columns=["embedding"])

In [ ]:
combined_df.head()

In [ ]:
# Limiting the amount of document used to 500 for this demo due to the rate limit on VoyageAI API
# Read more on VoyageAI rate limits: https://docs.voyageai.com/docs/rate-limits
max_documents = 500

if len(combined_df) > max_documents:
    combined_df = combined_df[:max_documents]

In [ ]:
import voyageai

vo = voyageai.Client(api_key=userdata.get("VOYAGE_API_KEY"))


def get_embedding(text: str) -> list[float]:
    if not text.strip():
        print("Attempted to get embedding for empty text.")
        return []

    embedding = vo.embed(text, model="voyage-large-2", input_type="document")

    return embedding.embeddings[0]


combined_df["embedding"] = combined_df["description"].apply(get_embedding)

combined_df.head()

## 第 2 步：创建数据库和集合

**要创建新的 MongoDB 数据库，请先搭建数据库集群：**
1. 注册[免费 MongoDB Atlas 账户](https://www.mongodb.com/cloud/atlas/register?utm_campaign=devrel&utm_source=community&utm_medium=cta&utm_content=Partner%20Cookbook&utm_term=richmond.alake)，已有账户的用户请[登录 MongoDB Atlas](https://account.mongodb.com/account/login?utm_campaign=devrel&utm_source=community&utm_medium=cta&utm_content=Partner%20Cookbook&utm_term=richmond.alake)
2. 在左侧窗格选择 “Database” 选项，页面会跳转到 Database Deployment 页，其中显示现有集群的部署规格。点击 “+Create” 按钮创建新的数据库集群。
3. 如需搭建数据库集群和获取 URI 的帮助，请参考我们关于搭建 MongoDB 集群并获取连接字符串的指南。
说明：创建概念验证环境时，别忘了把 Python 主机的 IP 或 0.0.0.0/0（任意 IP）加入白名单。
4. 成功创建并部署集群后，即可在 “Database Deployment” 页面访问该集群。
5. 点击集群的 “Connect” 按钮，查看通过多种语言驱动建立集群连接的选项。
6. 本教程只需要集群的 URI（唯一资源标识符）。获取该 URI，并将其复制到 Google Colabs 的 Secrets 环境中，变量名为 MONGO_URI，或放入 .env 文件等同类文件中。


创建集群后，转到集群页面，点击 + Create Database，在 MongoDB Atlas 集群中创建数据库和集合。数据库命名为 `tech_news`，集合命名为 `hacker_noon_tech_news`。



## 第 3 步：创建向量搜索索引

至此，你已经创建了集群、数据库和集合。

本节的步骤至关重要，它们确保能够用聊天机器人输入的查询，在 hacker_noon_tech_news 集合的记录中执行向量搜索。这一步的目标是创建向量搜索索引。具体做法请参考官方[向量搜索索引创建指南](https://www.mongodb.com/docs/atlas/atlas-vector-search/create-index/)。

在 MongoDB Atlas 上使用 JSON 编辑器创建向量搜索索引时，请确保向量搜索索引名为 vector_index，且向量搜索索引定义如下：

```
{
 "fields": [{
     "numDimensions": 1536,
     "path": "embedding",
     "similarity": "cosine",
     "type": "vector"
   }]
}

```

## 第 4 步：数据摄取

要把数据摄取到前面步骤创建的 MongoDB 数据库中，必须执行以下操作：
- 连接数据库和集合
- 清除集合中的现有记录
- 把数据集的 Pandas DataFrame 转换为字典，以便摄取
- 使用批次操作把字典摄取到 MongoDB

本教程需要集群的 URI（唯一资源标识符）。获取该 URI，并将其复制到 Google Colab Secrets 环境中，变量名为 MONGO_URI，或放入 .env 文件等同类文件中。


In [6]:
import pymongo
from google.colab import userdata


def get_mongo_client(mongo_uri):
    """Establish connection to the MongoDB."""
    try:
        client = pymongo.MongoClient(mongo_uri)
        print("Connection to MongoDB successful")
        return client
    except pymongo.errors.ConnectionFailure as e:
        print(f"Connection failed: {e}")
        return None


mongo_uri = userdata.get("MONGO_URI")
if not mongo_uri:
    print("MONGO_URI not set in environment variables")

mongo_client = get_mongo_client(mongo_uri)

DB_NAME = "tech_news"
COLLECTION_NAME = "hacker_noon_tech_news"

db = mongo_client[DB_NAME]
collection = db[COLLECTION_NAME]

Connection to MongoDB successful


In [7]:
# To ensure we are working with a fresh collection
# delete any existing records in the collection
collection.delete_many({})

DeleteResult({'n': 228012, 'electionId': ObjectId('7fffffff000000000000000e'), 'opTime': {'ts': Timestamp(1709660559, 7341), 't': 14}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1709660559, 7341), 'signature': {'hash': b'jT\xf1\xb4\xa9\xd3\xe3suu\x03`\x15(}\x8f\x00\x9f\xe9\x8a', 'keyId': 7320226449804230661}}, 'operationTime': Timestamp(1709660559, 7341)}, acknowledged=True)

In [ ]:
# Data Ingestion
combined_df_json = combined_df.to_dict(orient="records")
collection.insert_many(combined_df_json)

## 第 5 步：向量搜索

本节展示如何创建向量搜索自定义函数。该函数接受一个用户查询，对应用户在聊天机器人中的输入；它还接受第二个参数 `collection`，指向包含待搜索记录的数据库集合。

`vector_search` 函数基于 MongoDB 聚合管道中的一系列操作产生向量搜索结果。该管道包含 `$vectorSearch` 和 `$project` 阶段，根据用户查询的向量嵌入执行查询，然后格式化结果，省略后续处理不需要的记录属性。

下面的代码片段执行以下操作，以实现对电影的语义搜索：
1. 定义 `vector_search` 函数，它接受用户查询字符串和 MongoDB 集合作为输入，并基于向量相似度搜索返回与查询匹配的文档列表。
2. 调用先前定义的 `get_embedding` 函数为用户查询生成嵌入向量，该函数把查询字符串转换为向量表示。
3. 为 MongoDB 的 aggregate 函数构建管道，其中包含两个主要阶段：`$vectorSearch` 和 `$project`。
4. `$vectorSearch` 阶段执行真正的向量搜索。index 字段指定用于向量搜索的向量索引，其值应与前面步骤中向量搜索索引定义里填写的名称一致。queryVector 字段接收用户查询的嵌入向量表示。path 字段对应包含嵌入向量的文档字段。`numCandidates` 指定要考虑的候选文档数量，同时限制返回结果的数量。
5. $project 阶段格式化结果，排除 _id 和 `embedding` 字段。
6. aggregate 执行已定义的管道以获得向量搜索结果。最后一步操作把数据库返回的游标转换为列表。

In [9]:
def vector_search(user_query, collection):
    """
    Perform a vector search in the MongoDB collection based on the user query.

    Args:
    user_query (str): The user's query string.
    collection (MongoCollection): The MongoDB collection to search.

    Returns:
    list: A list of matching documents.
    """

    # Generate embedding for the user query
    query_embedding = get_embedding(user_query)

    if query_embedding is None:
        return "Invalid query or embedding generation failed."

    # Define the vector search pipeline
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 150,  # Number of candidate matches to consider
                "limit": 5,  # Return top 5 matches
            }
        },
        {
            "$project": {
                "_id": 0,  # Exclude the _id field
                "embedding": 0,  # Exclude the embedding field
                "score": {
                    "$meta": "vectorSearchScore"  # Include the search score
                },
            }
        },
    ]

    # Execute the search
    results = collection.aggregate(pipeline)
    return list(results)

## 第 6 步：用 Claude 3 模型处理用户查询

教程的最后一节按顺序执行以下操作：

- 接受以字符串形式给出的用户查询。
- 使用 VoyageAI 嵌入模型为用户查询生成嵌入向量。
- 加载 Anthropic 的 Claude，具体使用 `claude-opus-4-8` 模型，作为 RAG 系统的基础模型。
- 使用用户查询的嵌入向量执行向量搜索，从知识库中获取相关信息，为基础模型提供额外上下文。
- 把用户查询和收集到的补充信息一并提交给基础模型，生成响应。


需要特别注意的是，用户查询嵌入向量的维度必须与 MongoDB Atlas 上向量搜索索引定义中设置的维度一致。

本节的下一步是导入 anthropic 库并加载客户端，以使用 anthropic 处理消息和访问 Claude 模型的方法。请确保获取 Claude API 密钥，它位于 [Anthropic 官方网站](https://console.anthropic.com/settings/keys) 的设置页面中。


In [11]:
import anthropic

client = anthropic.Client(api_key=userdata.get("ANTHROPIC_API_KEY"))

下面更详细地说明下方代码片段中的操作：

1. 执行向量搜索：函数首先调用 `vector_search`，参数为用户查询和指定的集合。这在集合内执行搜索，利用向量嵌入查找与查询相关的信息。
2. 汇总搜索结果：`search_result` 初始化为空字符串，用于聚合搜索得到的信息。代码遍历 `vector_search` 函数返回的结果，把每条记录的详细信息（标题、公司名、URL、发布日期、文章 URL 和描述）格式化为便于阅读的字符串，并在每条记录末尾追加换行符 \n，把这些信息追加到 search_result。
3. 使用 Anthropic 客户端生成响应：随后函数构造对 Claude API 的请求（通过一个客户端对象，应当是先前创建的 anthropic. Client 类的实例）。请求指定了：
- 所用模型（`claude-opus-4-8`），即 Claude Opus 模型的版本。
- 生成响应的最大词元数限制（max_tokens=1024）。
- 一段系统说明，引导模型扮演可以访问科技公司文章和信息的“风险投资科技分析师”，并利用这些上下文提供建议。
- 实际交给模型处理的消息，把用户查询与聚合后的搜索结果作为上下文组合在一起。
4. 返回生成的响应和搜索结果：从响应 content 的第一个条目中提取响应文本，连同汇总后的搜索结果一并返回。


In [12]:
def handle_user_query(query, collection):
    get_knowledge = vector_search(query, collection)

    search_result = ""
    for result in get_knowledge:
        search_result += (
            f"Title: {result.get('title', 'N/A')}, "
            f"Company Name: {result.get('companyName', 'N/A')}, "
            f"Company URL: {result.get('companyUrl', 'N/A')}, "
            f"Date Published: {result.get('published_at', 'N/A')}, "
            f"Article URL: {result.get('url', 'N/A')}, "
            f"Description: {result.get('description', 'N/A')}, \n"
        )

    response = client.messages.create(
        model="claude-opus-4-8",
        max_tokens=1024,
        system="You are Venture Captital Tech Analyst with access to some tech company articles and information. You use the information you are given to provide advice.",
        messages=[
            {
                "role": "user",
                "content": "Answer this user query: "
                + query
                + " with the following context: "
                + search_result,
            }
        ],
    )

    return (response.content[0].text), search_result

本教程的最后一步是初始化查询，把它传入 `handle_user_query` 函数，并打印返回的响应。

In [13]:
# Conduct query with retrieval of sources
query = "Give me the best tech stock to invest in and tell me why"
response, source_information = handle_user_query(query, collection)

print(f"Response: {response}")
print(f"\\nSource Information: \\n{source_information}")

Response: Based on the information provided in the article titles and descriptions, Alibaba Group Holding Limited appears to be a top technology stock pick for 2023 according to renowned investor Ray Dalio. The article "Top 10 Technology Stocks to Buy in 2023 According to Ray Dalio" suggests that Alibaba is one of Dalio's favored tech investments for the year.

As a venture capital tech analyst, I would recommend considering an investment in Alibaba for the following reasons:

1. Endorsement from a respected investor: Ray Dalio, known for his successful investment strategies, has included Alibaba in his top 10 technology stock picks for 2023. His backing lends credibility to the investment potential of the company.

2. Strong market position: Alibaba is a leading e-commerce company in China with a significant market share. It has a diversified business model spanning e-commerce, cloud computing, digital media, and entertainment.

3. Growth potential: With China's large and growing midd